#Assignment 2

Max collaborators = 3

Hand holding is low for this assignment. Adjust accordingly. In case of confusion, feel free to reach out to your lab faculties. Good luck.

In [ ]:
COLLABORATORS_NAME = "Nasaba Saif Audree"
COLLABORATORS_ID = "22201612"

COLLABORATORS_NAME = "Mehrin Afroz Lopa"
COLLABORATORS_ID = "22299547"

COLLABORATORS_NAME = "Nisat Nisa"
COLLABORATORS_ID = "22299095"


#Loading dependencies

In [ ]:
import numpy as np
from tensorflow import keras
import sklearn
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report
from sklearn.utils.class_weight import compute_class_weight

#Loading dataset

In [ ]:
id = 23301489

(x_train, y_train), (x_test, y_test) = keras.datasets.cifar10.load_data()

x_train, y_train = sklearn.utils.resample(x_train, y_train, replace=False, n_samples=5000, random_state=id, stratify=y_train)
x_test, y_test = sklearn.utils.resample(x_test, y_test, replace=False, n_samples=1000, random_state=id, stratify=y_test)

#Task 1: Training a logistic regressor [10 Marks]

###1. Convert the images to grayscale

In [ ]:
def rgb_to_grayscale(images):
    return np.dot(images[..., :3], [0.299, 0.587, 0.114]).astype(np.float32)


x_train_gray = rgb_to_grayscale(x_train)
x_test_gray = rgb_to_grayscale(x_test)

### 2. Prepare the grayscale images for logistic regressor (reshape and normalize)

Use z-score normalization.

In [ ]:
n_train, h, w = x_train_gray.shape
x_train_lr = x_train_gray.reshape(n_train, h * w)
x_test_lr = x_test_gray.reshape(x_test_gray.shape[0], h * w)

train_mean = x_train_lr.mean(axis=0)
train_std = x_train_lr.std(axis=0)
train_std[train_std < 1e-8] = 1.0
x_train_lr = (x_train_lr - train_mean) / train_std
x_test_lr = (x_test_lr - train_mean) / train_std

###3. Create a validation set (20%)

In [ ]:
x_tr, x_val, y_tr, y_val = train_test_split(
    x_train_lr, y_train, test_size=0.2, random_state=id, stratify=y_train
)
y_tr_flat = y_tr.ravel()
y_val_flat = y_val.ravel()

###4. Compute class weights

Go through this link - https://scikit-learn.org/stable/modules/generated/sklearn.utils.class_weight.compute_class_weight.html

In [ ]:
classes = np.unique(y_tr_flat)
class_weights_arr = compute_class_weight(
    class_weight='balanced', classes=classes, y=y_tr_flat
)
class_weight_dict = dict(zip(classes, class_weights_arr))
print(class_weight_dict)

###5. Run the logistic regressor

Use L2 regularizer. Find the C hyperparameter value through grid search. Pick the testing values according to your understanding. Use at least 3 test values and at max 5. Use the computed class weights while training your model.

In [ ]:
param_grid = {'C': [0.001, 0.01, 0.1, 1.0, 10.0]}

log_reg_base = LogisticRegression(
    penalty='l2',
    solver='lbfgs',
    multi_class='multinomial',
    max_iter=5000,
    class_weight=class_weight_dict,
)

grid = GridSearchCV(
    log_reg_base, param_grid, cv=3, scoring='accuracy', n_jobs=-1, verbose=1
)
grid.fit(x_tr, y_tr_flat)

print('Best C:', grid.best_params_['C'])
print('Best CV accuracy:', grid.best_score_)

best_lr = grid.best_estimator_

###6. Evaluate the logistic regressor on the test set

In [ ]:
y_test_flat = y_test.ravel()
y_pred_lr = best_lr.predict(x_test_lr)
print('Test accuracy:', accuracy_score(y_test_flat, y_pred_lr))
print(classification_report(y_test_flat, y_pred_lr, digits=4))

###7. Write code to pick up a random image from the test set and display it

In [ ]:
rng = np.random.RandomState(id)
idx_lr = rng.randint(0, len(x_test_gray))
plt.imshow(x_test_gray[idx_lr], cmap='gray')
plt.title(f'Test index {idx_lr}')
plt.axis('off')
plt.show()

###8. Print the predicted class vs the original class

Do not print out the numerical class. Map it from here - https://keras.io/2/api/datasets/cifar10/

In [ ]:
CIFAR10_LABELS = [
    'airplane',
    'automobile',
    'bird',
    'cat',
    'deer',
    'dog',
    'frog',
    'horse',
    'ship',
    'truck',
]

sample = x_test_lr[idx_lr : idx_lr + 1]
pred = best_lr.predict(sample)[0]
true = y_test_flat[idx_lr]
print('Predicted:', CIFAR10_LABELS[pred])
print('Actual:   ', CIFAR10_LABELS[true])

#Task 2: Training a convolutional neural network [20 Marks]

We will not strictly control how you implement this code. You can use either Tensorflow or PyTorch. However the structure of the network must be -

```
Input -> Conv1 -> Conv2 -> Conv3 -> Fully Connected 1 -> Fully Connected 2 -> Output
```

Use activation functions and pooling as you want. Feel free to adjust dimensions as you need. Set the hyperparameters yourself. Use a maximum learning rate of 0.01 and a maximum epoch number of 100. Use AdamW as optimizer.

**Try achieving good accuracy. There are marks for that.**

###1. Train the model on grayscale images

You do not need to use the validation set. Train on the initial train set. Recalculate the class weights again and pass it to the optimizer function. Normalize the images before processing.

In [ ]:
keras.utils.set_random_seed(id)

y_train_flat = y_train.ravel()
y_test_flat = y_test.ravel()

classes_cnn = np.unique(y_train_flat)
cw_arr = compute_class_weight('balanced', classes=classes_cnn, y=y_train_flat)
cw_dict = dict(zip(classes_cnn, cw_arr))

x_train_g = x_train_gray[..., np.newaxis]
x_test_g = x_test_gray[..., np.newaxis]
g_mean, g_std = float(x_train_g.mean()), float(x_train_g.std())
g_std = g_std if g_std > 1e-8 else 1.0
x_train_g_n = (x_train_g - g_mean) / g_std
x_test_g_n = (x_test_g - g_mean) / g_std


def build_cnn_3conv(input_shape):
    return keras.Sequential(
        [
            keras.layers.Input(shape=input_shape),
            keras.layers.Conv2D(32, 3, padding='same', activation='relu'),
            keras.layers.BatchNormalization(),
            keras.layers.MaxPooling2D(2),
            keras.layers.Conv2D(64, 3, padding='same', activation='relu'),
            keras.layers.BatchNormalization(),
            keras.layers.MaxPooling2D(2),
            keras.layers.Conv2D(128, 3, padding='same', activation='relu'),
            keras.layers.BatchNormalization(),
            keras.layers.MaxPooling2D(2),
            keras.layers.Flatten(),
            keras.layers.Dense(256, activation='relu'),
            keras.layers.Dropout(0.5),
            keras.layers.Dense(128, activation='relu'),
            keras.layers.Dropout(0.3),
            keras.layers.Dense(10, activation='softmax'),
        ],
        name='cnn_3conv',
    )


model_g = build_cnn_3conv((32, 32, 1))
model_g.compile(
    optimizer=keras.optimizers.AdamW(learning_rate=0.001, weight_decay=1e-4),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy'],
)

cnn_callbacks = [
    keras.callbacks.EarlyStopping(
        monitor='loss', patience=12, restore_best_weights=True, min_delta=1e-4
    )
]

hist_g = model_g.fit(
    x_train_g_n,
    y_train_flat,
    epochs=100,
    batch_size=64,
    class_weight=cw_dict,
    callbacks=cnn_callbacks,
    verbose=1,
)

###2. Evaluate the model on the grayscale test set

In [ ]:
loss_g, acc_g = model_g.evaluate(x_test_g_n, y_test_flat, verbose=0)
print('Grayscale CNN test accuracy:', acc_g)

###3. Write code to pick up a random image from the test set and display it

In [ ]:
idx_cnn_g = rng.randint(0, len(x_test_g_n))
plt.imshow(x_test_g_n[idx_cnn_g].squeeze(), cmap='gray')
plt.title(f'Test index {idx_cnn_g}')
plt.axis('off')
plt.show()

###4. Print the predicted class vs the original class

In [ ]:
pred_cg = int(
    np.argmax(model_g.predict(x_test_g_n[idx_cnn_g : idx_cnn_g + 1], verbose=0), axis=1)
)
true_cg = int(y_test_flat[idx_cnn_g])
print('Predicted:', CIFAR10_LABELS[pred_cg])
print('Actual:   ', CIFAR10_LABELS[true_cg])

###5. Train the model on the RGB images

Keep the model same.

In [ ]:
x_train_f = x_train.astype(np.float32)
x_test_f = x_test.astype(np.float32)
rgb_mean, rgb_std = float(x_train_f.mean()), float(x_train_f.std())
rgb_std = rgb_std if rgb_std > 1e-8 else 1.0
x_train_rgb_n = (x_train_f - rgb_mean) / rgb_std
x_test_rgb_n = (x_test_f - rgb_mean) / rgb_std

keras.utils.set_random_seed(id + 7)
model_rgb = build_cnn_3conv((32, 32, 3))
model_rgb.compile(
    optimizer=keras.optimizers.AdamW(learning_rate=0.001, weight_decay=1e-4),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy'],
)
hist_rgb = model_rgb.fit(
    x_train_rgb_n,
    y_train_flat,
    epochs=100,
    batch_size=64,
    class_weight=cw_dict,
    callbacks=cnn_callbacks,
    verbose=1,
)

###6. Evaluate the model on the RGB images

In [ ]:
loss_rgb, acc_rgb = model_rgb.evaluate(x_test_rgb_n, y_test_flat, verbose=0)
print('RGB CNN test accuracy:', acc_rgb)

###7. Write code to pick up a random image from the test set and display it

In [ ]:
idx_cnn_rgb = rng.randint(0, len(x_test_rgb_n))
plt.imshow(x_test[idx_cnn_rgb])
plt.title(f'Test index {idx_cnn_rgb}')
plt.axis('off')
plt.show()

###8. Print the predicted class vs the original class

In [ ]:
pred_cr = int(
    np.argmax(
        model_rgb.predict(x_test_rgb_n[idx_cnn_rgb : idx_cnn_rgb + 1], verbose=0), axis=1
    )
)
true_cr = int(y_test_flat[idx_cnn_rgb])
print('Predicted:', CIFAR10_LABELS[pred_cr])
print('Actual:   ', CIFAR10_LABELS[true_cr])

###9. Add one more conv layer and check if it increases the performance for the RGB images

In [ ]:
def build_cnn_4conv(input_shape):
    return keras.Sequential(
        [
            keras.layers.Input(shape=input_shape),
            keras.layers.Conv2D(32, 3, padding='same', activation='relu'),
            keras.layers.BatchNormalization(),
            keras.layers.MaxPooling2D(2),
            keras.layers.Conv2D(64, 3, padding='same', activation='relu'),
            keras.layers.BatchNormalization(),
            keras.layers.MaxPooling2D(2),
            keras.layers.Conv2D(128, 3, padding='same', activation='relu'),
            keras.layers.BatchNormalization(),
            keras.layers.MaxPooling2D(2),
            keras.layers.Conv2D(256, 3, padding='same', activation='relu'),
            keras.layers.BatchNormalization(),
            keras.layers.MaxPooling2D(2),
            keras.layers.Flatten(),
            keras.layers.Dense(256, activation='relu'),
            keras.layers.Dropout(0.5),
            keras.layers.Dense(128, activation='relu'),
            keras.layers.Dropout(0.3),
            keras.layers.Dense(10, activation='softmax'),
        ],
        name='cnn_4conv',
    )


keras.utils.set_random_seed(id + 13)
model_rgb4 = build_cnn_4conv((32, 32, 3))
model_rgb4.compile(
    optimizer=keras.optimizers.AdamW(learning_rate=0.001, weight_decay=1e-4),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy'],
)
hist_rgb4 = model_rgb4.fit(
    x_train_rgb_n,
    y_train_flat,
    epochs=100,
    batch_size=64,
    class_weight=cw_dict,
    callbacks=cnn_callbacks,
    verbose=1,
)
loss4, acc4 = model_rgb4.evaluate(x_test_rgb_n, y_test_flat, verbose=0)
print(f'3-conv RGB test accuracy: {acc_rgb:.4f}')
print(f'4-conv RGB test accuracy: {acc4:.4f}')

###10. Use the next markup to write down your observations and reasonings on the performances of the models

Observations

1. **Logistic regression on flattened pixels** treats each image as a 1,024-dimensional vector with no spatial structure. Accuracy stays well below the CNNs because the model cannot exploit local edges, textures, or translation patterns that convolutions capture.

2. **Grayscale vs RGB for the same 3-conv architecture** — RGB inputs usually perform better on CIFAR-10 because many classes are easier to separate with color cues (e.g., sky vs foliage, vehicle paint vs natural objects). Grayscale discards that signal, so the network relies more on shape and texture alone.

3. **Effect of a fourth convolutional block** — Adding depth can improve representation power, but on a small subset (5,000 train / 1,000 test) it can also increase overfitting or make optimization harder if capacity grows without enough data or regularization. Whether accuracy goes up or down compared to the 3-conv model depends on the run; comparing the printed test accuracies answers the assignment question empirically.

4. **Class weights** — Balanced weights upweight under-represented classes in the subsample, which often helps macro-averaged performance and stabilizes training when class counts in the subset are still uneven.

5. **Normalization** — Z-score (logistic and CNN) centers scales so gradients and regularization behave more predictably than with raw \[0, 255\] counts alone.